In [ ]:
# Cell 1 — Install dependencies and imports

!pip -q install -U unsloth transformers datasets "trl>=0.12.0,<0.15.0" accelerate bitsandbytes sentencepiece  

import os
import json
import torch
from pathlib import Path
from datasets import load_dataset
from unsloth import FastLanguageModel
from datetime import datetime, timezone
from transformers import Trainer, TrainingArguments
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig, DataCollatorForCompletionOnlyLM

print("✓ All imports successful")

In [ ]:
# Cell 2 — Training config

TRAIN_CONFIG = {
    "run_name": "sprint51_task1_unsloth_llama32_1b_en2zh_full",
    "seed": 3407,
    "base_model": "unsloth/Llama-3.2-1B-bnb-4bit",

    "train_file": "/kaggle/input/datasets/abdighaz/en-cmn-sprint51/mandarin_task1_dataset/data/processed/finetune_train.jsonl",
    "prompt_validation_file": "/kaggle/input/datasets/abdighaz/en-cmn-sprint51/mandarin_task1_dataset/data/processed/prompt_validation.jsonl",
    "data_manifest_path": "/kaggle/input/datasets/abdighaz/en-cmn-sprint51/mandarin_task1_dataset/manifests/data_manifest.json",
    "hardware_probe_path": "/kaggle/input/datasets/abdighaz/en-cmn-sprint51/mandarin_task1_dataset/manifests/hardware_probe.json",

    "output_dir": "/kaggle/working/task1_en2zh_outputs",
    "checkpoint_dir": "/kaggle/working/task1_en2zh_outputs/checkpoints",
    "final_dir": "/kaggle/working/task1_en2zh_outputs/final",
    "manifest_dir": "/kaggle/working/task1_en2zh_outputs/manifests",

    "max_seq_length": 1024,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "eval_accumulation_steps": 1,

    "learning_rate": 5e-5,
    "num_train_epochs": 2,  
    "logging_steps": 25,     
    "save_steps": 500,       
    "eval_steps": 500,       
    "save_total_limit": 3,  
    "max_grad_norm": 1.0,

    "optim": "adamw_8bit",
    "weight_decay": 0.01,
    "warmup_steps": 200,
    "warmup_ratio": 0.05,
    "lr_scheduler_type": "cosine",

    "load_in_4bit": True,
    "lora_r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "max_steps": -1,

    "generation_prompt": "Translate to Mandarin:\nThe weather is nice today."
}

In [ ]:
# Cell 3 — Prepare directories and load dataset

for p in [
    TRAIN_CONFIG["output_dir"],
    TRAIN_CONFIG["checkpoint_dir"],
    TRAIN_CONFIG["final_dir"],
    TRAIN_CONFIG["manifest_dir"],
]:
    Path(p).mkdir(parents=True, exist_ok=True)

print("Output dir:", TRAIN_CONFIG["output_dir"])

data_files = {
    "train": TRAIN_CONFIG["train_file"],
    "validation": TRAIN_CONFIG["prompt_validation_file"],
}

raw_datasets = load_dataset("json", data_files=data_files)

# raw_datasets["train"] = raw_datasets["train"].filter(
#     lambda x: x["source_lang"] == "en" and x["target_lang"] == "zh"
# )
# raw_datasets["validation"] = raw_datasets["validation"].filter(
#     lambda x: x["source_lang"] == "en" and x["target_lang"] == "zh"
# )

# print(raw_datasets)
# print("Train rows:", len(raw_datasets["train"]))
# print("Validation rows:", len(raw_datasets["validation"]))

# ---------- Subsample to avoid Kaggle timeout ----------
# Adjust these two numbers according to how much time you have
TRAIN_SUBSET_SIZE   = 40000   # ~40k examples ≈ good balance
VAL_SUBSET_SIZE     = 1500     # 1500 keep validation reasonably large

# Shuffle + select (seed for reproducibility)
raw_datasets["train"] = (
    raw_datasets["train"]
    .shuffle(seed=TRAIN_CONFIG["seed"])
    .select(range(min(TRAIN_SUBSET_SIZE, len(raw_datasets["train"]))))
)

raw_datasets["validation"] = (
    raw_datasets["validation"]
    .shuffle(seed=TRAIN_CONFIG["seed"])
    .select(range(min(VAL_SUBSET_SIZE, len(raw_datasets["validation"]))))
)

print("After subsampling:")
print("Train rows:", len(raw_datasets["train"]))
print("Validation rows:", len(raw_datasets["validation"]))

In [ ]:
# Cell 4 — Load model and tokenizer

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = TRAIN_CONFIG["base_model"],
    max_seq_length = TRAIN_CONFIG["max_seq_length"],
    dtype = None,
    load_in_4bit = TRAIN_CONFIG["load_in_4bit"],
)


model = FastLanguageModel.get_peft_model(
    model,
    r = TRAIN_CONFIG["lora_r"],                 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = TRAIN_CONFIG["lora_alpha"],    
    lora_dropout = TRAIN_CONFIG["lora_dropout"],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = TRAIN_CONFIG["seed"],
    use_rslora = False,                         
)


tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

print("Model and tokenizer loaded.")

In [ ]:
# Cell 5 — Format dataset

def formatting_prompts_func(examples):
    sources = examples["source_text"]
    targets = examples["target_text"]
    texts = []

    eos_token = tokenizer.eos_token

    for src, tgt in zip(sources, targets):
        convo = [
            {
                "role": "system",
                "content": (
                    "You are an expert translator for English to Mandarin Chinese (普通話). "
                    "Rules you must follow:\n"
                    "1. Output ONLY the Mandarin translation in Simplified Chinese characters.\n"
                    "2. Use standard Mainland China Mandarin vocabulary and grammar.\n"
                    "3. Do NOT use Cantonese (廣東話) vocabulary or particles.\n"
                    "4. Do NOT romanize (no Pinyin).\n"
                    "5. Do NOT explain, repeat the source, or add any commentary.\n"
                    "6. If a proper noun has no Mandarin equivalent, keep the original English term."
                )
            },
            {
                "role": "user",
                "content": f"Translate to Mandarin:\n{src}"
            },
            {
                "role": "assistant",
                "content": tgt
            }
        ]

        formatted_text = tokenizer.apply_chat_template(
            convo,
            tokenize = False,
            add_generation_prompt = False
        )

        if not formatted_text.endswith(eos_token):
            formatted_text += eos_token

        texts.append(formatted_text)

    return {"text": texts}


dataset = raw_datasets.map(
    formatting_prompts_func,
    batched = True,
    remove_columns = raw_datasets["train"].column_names 
)

# 5. Quick Verification
print(f"✅ Row 1 Sample: {dataset['train'][0]['text'][-20:]}") # Should show the EOS tag


In [ ]:
# Cell 6: Tokenization & Data Collation

# 1. Define the Tokenization Logic
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation = True,
        max_length = TRAIN_CONFIG["max_seq_length"],   
        add_special_tokens = False,
    )

# 2. Map tokenization (Removed manual padding here, the collator handles it better)
tokenized_dataset = dataset.map(
    tokenize_function,
    batched = True,
    num_proc = 2,
    remove_columns = dataset["train"].column_names
)

# 3. Final Setup
tokenizer.padding_side = "right"

# 4. Initialize the CORRECT Data Collator
# This is what forces the model to ignore the English prompt and only learn the Chinese + EOS
response_template = "<|im_start|>assistant\n"
data_collator = DataCollatorForCompletionOnlyLM(
    response_template = response_template,
    tokenizer = tokenizer
)

# --- Quick Verification ---
print("✅ Tokenization Complete!")
print(f"Features available: {tokenized_dataset['train'].column_names}")

In [ ]:
# Cell 7 — Training arguments and SFTTrainer

training_args = SFTConfig(
    output_dir = TRAIN_CONFIG["checkpoint_dir"],
    per_device_train_batch_size = TRAIN_CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size = TRAIN_CONFIG["per_device_eval_batch_size"],
    gradient_accumulation_steps = TRAIN_CONFIG["gradient_accumulation_steps"],
    num_train_epochs = TRAIN_CONFIG["num_train_epochs"],
    max_steps = TRAIN_CONFIG.get("max_steps", -1),
    learning_rate = TRAIN_CONFIG["learning_rate"],
    warmup_ratio = TRAIN_CONFIG["warmup_ratio"],
    weight_decay = TRAIN_CONFIG["weight_decay"],
    lr_scheduler_type = TRAIN_CONFIG["lr_scheduler_type"],
    max_grad_norm = TRAIN_CONFIG["max_grad_norm"],
    logging_steps = TRAIN_CONFIG["logging_steps"],
    save_strategy = "steps",
    save_steps = TRAIN_CONFIG["save_steps"],
    eval_strategy = "steps",
    eval_steps = TRAIN_CONFIG["eval_steps"],
    save_total_limit = TRAIN_CONFIG["save_total_limit"],
    load_best_model_at_end = True,
    metric_for_best_model = "eval_loss",
    greater_is_better = False,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    optim = TRAIN_CONFIG["optim"],
    seed = TRAIN_CONFIG["seed"],
    report_to = "none",
    dataset_text_field = "text",          # important
    max_seq_length = TRAIN_CONFIG["max_seq_length"],
    packing = False,                      # keep False for translation
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],     # use the non-tokenized version
    eval_dataset = dataset["validation"],
    args = training_args,
    # data_collator can still be used if you want completion-only
)


# training_args = TrainingArguments(
#     output_dir = TRAIN_CONFIG["checkpoint_dir"],
#     run_name = TRAIN_CONFIG["run_name"],
#     per_device_train_batch_size = TRAIN_CONFIG["per_device_train_batch_size"],
#     per_device_eval_batch_size = TRAIN_CONFIG["per_device_eval_batch_size"],
#     gradient_accumulation_steps = TRAIN_CONFIG["gradient_accumulation_steps"],
#     average_tokens_across_devices = False,
#     eval_accumulation_steps = TRAIN_CONFIG["eval_accumulation_steps"],
#     max_grad_norm = TRAIN_CONFIG["max_grad_norm"],
#     num_train_epochs = TRAIN_CONFIG["num_train_epochs"],
#     learning_rate = TRAIN_CONFIG["learning_rate"],
#     warmup_ratio = 0.05,
#     weight_decay = TRAIN_CONFIG["weight_decay"],
#     lr_scheduler_type = TRAIN_CONFIG["lr_scheduler_type"],
#     logging_steps = TRAIN_CONFIG["logging_steps"],
#     save_strategy = "steps",
#     label_names = ["labels"],
#     max_steps = TRAIN_CONFIG["max_steps"],
#     save_steps = TRAIN_CONFIG["save_steps"],
#     save_total_limit = TRAIN_CONFIG["save_total_limit"],
#     eval_strategy = "steps",
#     eval_steps = TRAIN_CONFIG["eval_steps"],
#     fp16 = not torch.cuda.is_bf16_supported(),
#     bf16 = torch.cuda.is_bf16_supported(),
#     gradient_checkpointing = True,
#     optim = TRAIN_CONFIG["optim"],
#     remove_unused_columns = False,
#     load_best_model_at_end = True,
#     metric_for_best_model = "eval_loss",
#     report_to = "none",
#     seed = TRAIN_CONFIG["seed"],
#     dataset_text_field = "text",
#     max_seq_length = TRAIN_CONFIG["max_seq_length"],
#     packing = False,
# )

# trainer = Trainer(
#     model = model,
#     train_dataset = tokenized_dataset["train"],
#     eval_dataset = tokenized_dataset["validation"],
#     data_collator = data_collator,
#     args = training_args,
# )


In [ ]:
# Cell 8 — Train, save, reload, smoke test, manifest

train_result = trainer.train()
print(train_result)

trainer.save_model(TRAIN_CONFIG["final_dir"])
tokenizer.save_pretrained(TRAIN_CONFIG["final_dir"])

# Optional merged save
model.save_pretrained_merged(
    TRAIN_CONFIG["final_dir"] + "_merged_16bit",
    tokenizer,
    save_method = "merged_16bit"
)

print("Saved final model artifacts to:", TRAIN_CONFIG["final_dir"])

reload_model, reload_tokenizer = FastLanguageModel.from_pretrained(
    model_name = TRAIN_CONFIG["final_dir"],
    max_seq_length = TRAIN_CONFIG["max_seq_length"],
    dtype = None,
    load_in_4bit = TRAIN_CONFIG["load_in_4bit"],
)

# Make sure the tokenizer has a chat template
if getattr(reload_tokenizer, "chat_template", None) is None:
    print("WARNING: chat_template missing after reload — re-applying ChatML")
    reload_tokenizer = get_chat_template(
        reload_tokenizer,
        chat_template = "chatml",
    )
else:
    print("chat_template present after reload (as expected)")

FastLanguageModel.for_inference(reload_model)

# ---------- Smoke test ----------
test_messages = [
    {
        "role": "system",
        "content": (
            "You are an expert translator for English to Mandarin Chinese (普通話). "
            "Output ONLY the Mandarin translation in Simplified Chinese."
        )
    },
    {
        "role": "user",
        "content": "Translate to Mandarin:\nThe weather is nice today."
    }
]

test_prompt = reload_tokenizer.apply_chat_template(
    test_messages,
    tokenize = False,
    add_generation_prompt = True
)

inputs = reload_tokenizer(test_prompt, return_tensors="pt").to(reload_model.device)

with torch.no_grad():
    outputs = reload_model.generate(
        **inputs,
        max_new_tokens = 64,
        do_sample = False,
        use_cache = True
    )

decoded = reload_tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Prompt:", test_prompt)
print("Generation:", decoded)

with open(TRAIN_CONFIG["data_manifest_path"], "r", encoding="utf-8") as f:
    data_manifest = json.load(f)

with open(TRAIN_CONFIG["hardware_probe_path"], "r", encoding="utf-8") as f:
    hardware_probe = json.load(f)

candidate_manifest = {
    "run_name": TRAIN_CONFIG["run_name"],
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "model": {
        "base_model": TRAIN_CONFIG["base_model"]
    },
    "training": {
        "method_selected": "unsloth_lora_4bit",
        "seed": TRAIN_CONFIG["seed"],
        "learning_rate": TRAIN_CONFIG["learning_rate"],
        "num_train_epochs": TRAIN_CONFIG["num_train_epochs"],
        "per_device_train_batch_size": TRAIN_CONFIG["per_device_train_batch_size"],
        "gradient_accumulation_steps": TRAIN_CONFIG["gradient_accumulation_steps"],
        "max_seq_length": TRAIN_CONFIG["max_seq_length"],
        "language_direction_trained": "en->zh"
    },
    "artifacts": {
        "output_dir": TRAIN_CONFIG["output_dir"],
        "checkpoint_dir": TRAIN_CONFIG["checkpoint_dir"],
        "final_dir": TRAIN_CONFIG["final_dir"],
        "merged_16bit_dir": TRAIN_CONFIG["final_dir"] + "_merged_16bit"
    },
    "data": {
        "dataset_version": data_manifest["dataset_version"],
        "language_directions": data_manifest["language_directions"],
        "files": data_manifest["files"]
    },
    "hardware_probe": hardware_probe,
    "smoke_test": {
        "checkpoint_load_success": True,
        "prompt": "Translate to Mandarin: The weather is nice today.",
        "generation": decoded
    }
}

candidate_manifest_path = Path(TRAIN_CONFIG["manifest_dir"]) / "candidate_manifest.json"
with open(candidate_manifest_path, "w", encoding="utf-8") as f:
    json.dump(candidate_manifest, f, indent=2, ensure_ascii=False)

print("Saved:", candidate_manifest_path)
print(json.dumps(candidate_manifest, indent=2, ensure_ascii=False))

print("\nOutput files:")
for p in Path(TRAIN_CONFIG["output_dir"]).rglob("*"):
    print(p)